# 01 — Dataset acquisition and export

This notebook runs the existing `guesser-data` command-line workflow: **download → prepare → optional preview → export**. The implementation stays in `models/guesser/data/src/`; this notebook only supplies options and calls the public CLI.

> **Production caution:** downloading all Quick, Draw! categories reads the full source files and can take a long time. Do not enable the pipeline while another data command is running; let the active job finish rather than starting a second one here.

The data package uses Python 3.13 while this Jupyter workspace uses Python 3.12, so importing its implementation modules into this kernel would cross the project's environment boundary. Calling the existing CLI through `uv` reuses the exact implementation without copying its logic. The next code cell can run `uv sync --project models/guesser/data --frozen` from the repository root. Keep its opt-in disabled while any data command is active; syncing may update packages in the live environment.

In [ ]:
from pathlib import Path
import subprocess

from guesser_evaluation.notebook_tools import module_source_path

# Resolve project paths from the installed workspace, independent of the notebook cwd.
EVALUATION_PROJECT = module_source_path("notebook_tools.py").parents[2]
REPOSITORY_ROOT = EVALUATION_PROJECT.parents[2]
DATA_PROJECT = EVALUATION_PROJECT.parent / "data"

# Dependency sync can change packages used by a live downloader, so require an opt-in.
SYNC_DATA_ENVIRONMENT = False
if SYNC_DATA_ENVIRONMENT:
    # Run the documented setup command from the repository root.
    subprocess.run(
        ["uv", "sync", "--project", "models/guesser/data", "--frozen"],
        cwd=REPOSITORY_ROOT,
        check=True,
    )
else:
    print("Environment sync skipped; set SYNC_DATA_ENVIRONMENT = True to run it.")

## Options

Change the corresponding values in the next code cell. `None`/empty values leave defaults with the data package rather than recreating them here. Source defaults are in [`settings.py`](../../data/src/settings.py); exact command flags are shown by `guesser-data <command> --help`.

| Option | Notebook default | Choices / effect |
|---|---|---|
| `PROFILE` (`--profile`) | `production` | `production` (345 labels) or `smoke` (reduced sample/split counts; downloads still use the source). |
| `DATA_ROOT` (`--data-root`) | `None` | `None` uses the package default `.local/data`; otherwise provide the data-root directory (the profile subdirectory is added by the CLI). |
| `CONCURRENCY` (`--concurrency`) | `None` | `None` uses the package default of 2; the download CLI accepts 1, 2, or 3. |
| `CANDIDATE_RESERVE` (`--candidates`) | `None` | `None` uses the profile default (5,500 per production class, 30 per smoke class); set an integer only to override it. |
| `CATEGORY_FILTER` (`--class`) | empty tuple | Empty means all official categories; list category names to request only those categories. |
| `INCLUDE_PREVIEW` | `False` | If enabled, create the optional training-only preview between prepare and export. |
| `RUN_PIPELINE` | `False` | Must be set to `True` explicitly before the production steps execute. Run the full sequence only when no other data process is active. |

Export audits the prepared data and writes separate trainer and held-out-test ZIPs. With the default production options, the trainer archive is `models/guesser/data/.local/data/production/exports/trainer-production.zip`; changing `DATA_ROOT` or `PROFILE` changes that path. Keep the held-out-test package sealed until the model and evaluation protocol are frozen.

In [ ]:
# Set only intentional overrides; None delegates defaults to the data CLI.
DATA_ROOT = None
PROFILE = "production"
CONCURRENCY = None
CANDIDATE_RESERVE = None
CATEGORY_FILTER = ()
INCLUDE_PREVIEW = False
RUN_PIPELINE = False

print(f"Data project: {DATA_PROJECT}")
print(f"Repository root: {REPOSITORY_ROOT}")

## Run the existing data commands

The small adapter below invokes the package's CLI with its locked Python 3.13 environment. It does not implement downloading, cleaning, splitting, auditing, or packaging. `subprocess.run(..., check=True)` streams command output into the notebook and stops the sequence on the first failure.

With `SYNC_DATA_ENVIRONMENT = False` and `RUN_PIPELINE = False`, running all notebook cells is safe: it skips dependency changes and production work, then reports status and the expected export path. Change either flag only when you deliberately want that operation and no data job is active.

In [ ]:
def run_data_command(command: str, *arguments: str) -> None:
    """Run one existing guesser-data command in its locked project environment."""
    # Let uv select the data project's Python 3.13 environment without syncing mid-run.
    command_line = [
        "uv",
        "run",
        "--project",
        str(DATA_PROJECT),
        "--frozen",
        "--no-sync",
        "guesser-data",
        command,
    ]
    # Pass explicit path/profile overrides while leaving other defaults to the CLI.
    if DATA_ROOT is not None:
        command_line.extend(["--data-root", str(Path(DATA_ROOT).expanduser())])
    command_line.extend(["--profile", PROFILE, *arguments])
    # Inherit output so long-running downloads and actionable failures stay visible.
    subprocess.run(command_line, cwd=REPOSITORY_ROOT, check=True)

In [ ]:
# Status is read-only; it shows whether the selected profile is ready for the next stage.
run_data_command("status")

# Report where the export step will write packages for the selected root and profile.
DEFAULT_DATA_ROOT = DATA_PROJECT / ".local" / "data"
EFFECTIVE_DATA_ROOT = Path(DATA_ROOT).expanduser() if DATA_ROOT is not None else DEFAULT_DATA_ROOT
EXPORT_DIRECTORY = EFFECTIVE_DATA_ROOT / PROFILE / "exports"
print(f"Expected trainer package after export: {EXPORT_DIRECTORY / f'trainer-{PROFILE}.zip'}")
print(f"Expected held-out package after export: {EXPORT_DIRECTORY / f'held-out-test-{PROFILE}.zip'}")

In [ ]:
if not RUN_PIPELINE:
    # Keep opening or running all cells from starting expensive production work.
    print("Pipeline disabled. Set RUN_PIPELINE = True only when ready to run it.")
else:
    download_options = []
    if CONCURRENCY is not None:
        download_options.extend(["--concurrency", str(CONCURRENCY)])
    if CANDIDATE_RESERVE is not None:
        download_options.extend(["--candidates", str(CANDIDATE_RESERVE)])
    for category in CATEGORY_FILTER:
        download_options.extend(["--class", category])

    # Run the package's established stages in documented dependency order.
    run_data_command("download", *download_options)
    run_data_command("prepare")
    if INCLUDE_PREVIEW:
        run_data_command("preview")
    run_data_command("export")
    run_data_command("status")

## Handoff

The trainer ZIP contains the training/validation package; the held-out-test ZIP is a distinct export and is not for tuning. The audit in [`02_dataset_audit.ipynb`](02_dataset_audit.ipynb) checks prepared manifests, labels, split counts, and cross-split identity isolation. Neither notebook evaluates recognition quality; that requires a frozen model checkpoint and predictions.